In [1]:
import string
import pandas as pd
import nltk
from datasets import load_dataset
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize, word_tokenize
from sklearn.feature_extraction.text import CountVectorizer

# Download the NLTK resources used in this notebook.
nltk.download('stopwords')
nltk.download('punkt')
# Newer NLTK versions may also require punkt_tab.
nltk.download('punkt_tab')

/home/codespace/.python/current/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /home/codespace/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
# Task 0: Data Preparation

dataset = load_dataset('stanfordnlp/imdb', split='train')

df = pd.DataFrame(dataset)
df = df.rename(columns={'text': 'review_text', 'label': 'sentiment'})
df = df[['review_text', 'sentiment']]

sentiment_map = {0: 'negative', 1: 'positive'}
df['sentiment'] = df['sentiment'].map(sentiment_map)

print("Dataset shape:", df.shape)
print("Missing values:\n", df.isna().sum())
print(df.head())

df.to_csv('movie_reviews.csv', index=False, encoding='utf-8')
print("Saved movie_reviews.csv")

Generating unsupervised split: 100%|██████████| 50000/50000 [00:00<00:00, 224654.98 examples/s]


Dataset shape: (25000, 2)
Missing values:
 review_text    0
sentiment      0
dtype: int64
                                         review_text sentiment
0  I rented I AM CURIOUS-YELLOW from my video sto...  negative
1  "I Am Curious: Yellow" is a risible and preten...  negative
2  If only to avoid making this type of film in t...  negative
3  This film was probably inspired by Godard's Ma...  negative
4  Oh, brother...after hearing about this ridicul...  negative
Saved movie_reviews.csv


In [ ]:
# Task 1: Text Preprocessing

# Get the English stop words from NLTK.
stop_words = set(stopwords.words('english'))

# Create a translation table that removes punctuation.
punctuation_table = str.maketrans('', '', string.punctuation)

def preprocess_text(text):
    # Handle missing values safely.
    if pd.isna(text):
        return ''

    # a. Convert to lowercase.
    text = text.lower()

    # b. Remove punctuation marks.
    text = text.translate(punctuation_table)

    # c. Remove stop words.
    words = text.split()
    words = [word for word in words if word not in stop_words]

    return ' '.join(words)

df['processed_text'] = df['review_text'].apply(preprocess_text)

print(df[['review_text', 'processed_text']].head(3).to_string(index=False))

### Task 1

**1. What is the purpose of text preprocessing in NLP applications?**
The purpose of text preprocessing in NLP applications is to clean and normalize raw text data before feeding it into a model or algorithm. This process helps reduce noise and eliminate unnecessary elements (such as punctuation, stop words, and inconsistent capitalization).

**2. How does stop word removal affect text analysis?**
It helps reduce data size and the number of features, thereby accelerating processing and model training.

It enables the model to focus on words that carry significant meaning.

**3. Why is case normalization important in text preprocessing?**
It helps standardize word representation.

It reduces vocabulary size, helping the model learn more effectively and mitigating data sparsity.

**4. What are potential drawbacks of removing stop words?**
It can lead to the loss of important grammatical and semantic information, particularly in negative statements (e.g., "not good"), questions, or comparative structures.

In tasks such as sentiment analysis, machine translation, and text summarization, stop words are sometimes essential for correctly understanding the meaning of a sentence.

Removing stop words can compromise natural expression if the output requires complete, coherent text.

In [ ]:
# Task 2: Text Tokenization

# a. Sentence tokenization.
df['tokenized_sentences'] = df['review_text'].apply(
    lambda text: sent_tokenize(text) if isinstance(text, str) else []
)

# b. Word tokenization.
df['tokenized_words'] = df['review_text'].apply(
    lambda text: word_tokenize(text) if isinstance(text, str) else []
)
print(df[['tokenized_sentences', 'tokenized_words']].head(3).to_string(index=False))

### Task 2

**1. What is the difference between word and sentence tokenization?**
Word tokenization: Splitting text into smaller units known as words or tokens (e.g., "I love USA" → ["I", "love", "USA"]). The output units are individual words.
Sentence tokenization: Splitting text into complete sentences (e.g., "I love USA. It is Freedom." → ["I love USA.", "It is freedom."]). The output units are sentences.

**2. Why is tokenization important for text analysis?**
Converts raw text (character strings) into meaningful units (tokens) that machines can process.
Serves as the fundamental input for most subsequent NLP tasks, such as POS tagging, NER, parsing, and vectorization (Bag-of-Words, TF-IDF, embeddings).
Helps standardize and structure text data, facilitating statistical analysis, machine learning, and deep learning.

**3. How do regular expressions help in tokenization?**
Define patterns to identify tokens, such as words, numbers, emails, URLs, hashtags, and mentions.
Allow text splitting based on flexible rules rather than relying solely on whitespace or simple punctuation.
Regex is particularly useful for non-standard text like social media content, which contains many special symbols, abbreviations, and emojis.

**4. What challenges might you encounter when tokenizing social media text?**
Abbreviations and slang: "btw", "lol", "idk", etc.
Emojis and emoticons: 😊
URLs and emails
Spelling errors, random capitalization

In [ ]:
# Task 3: Basic Text Vectorization

# a. Create a Bag-of-Words matrix from the processed text.
vectorizer = CountVectorizer(max_features=1000)
bow_matrix = vectorizer.fit_transform(df['processed_text'].fillna(''))

# b. Calculate total frequency for each word across all reviews.
word_frequencies = bow_matrix.sum(axis=0).A1
vocabulary = vectorizer.get_feature_names_out()

# c. Sort words by frequency and display the top 20.
frequency_pairs = list(zip(vocabulary, word_frequencies))
top_20_words = sorted(frequency_pairs, key=lambda item: item[1], reverse=True)[:20]

top_words_df = pd.DataFrame(top_20_words, columns=['word', 'frequency'])
print(top_words_df.to_string(index=False))

  word  frequency
    br      57145
 movie      41813
  film      37461
   one      25511
  like      19645
  good      14555
  even      12506
 would      12135
  time      11782
really      11663
 story      11455
   see      11223
  much       9586
  well       9378
   get       9212
  also       9073
people       8952
   bad       8914
 great       8894
 first       8859


### Task 3 

**1. What information does a bag-of-words model capture?**
The Bag-of-Words (BoW) model captures the frequency of words within a text, disregarding word order and grammar. It represents each document as a vector, where each dimension corresponds to a word in the vocabulary.

The value at each dimension represents the number of times the corresponding word appears in the text (or a binary 0/1 value if `binary=True` is used).

**2. How does vectorization help in text analysis?**
It enables processing by computers and machine learning algorithms.

It allows for the application of algebraic operations, statistical methods, and machine learning models

It standardizes text data into a structured format, making it easier to compare documents and calculate distances or similarities between them.

**3. What are the limitations of basic frequency counts?**
Does not distinguish between important words and common words.

Does not capture semantics.

Results in high-dimensional, sparse vectors: with a large vocabulary, vectors become very long and consist mostly of zeros, leading to high memory and computational costs.

Cannot handle out-of-vocabulary (OOV) words: words not present in the vocabulary are ignored.

In [ ]:
# Task 4: Save and Document Results

comparison_df = pd.DataFrame({
    'original_text': df['review_text'],
    'processed_text': df['processed_text'],
    'number_of_sentences': df['tokenized_sentences'].apply(len),
    'words_before_preprocessing': df['tokenized_words'].apply(len),
    'words_after_preprocessing': df['processed_text'].apply(
        lambda text: len(text.split()) if isinstance(text, str) and text else 0
    )
})

print("Average number of sentences per review:",
      comparison_df['number_of_sentences'].mean())
print("Average word count before preprocessing:",
      comparison_df['words_before_preprocessing'].mean())
print("Average word count after preprocessing:",
      comparison_df['words_after_preprocessing'].mean())

comparison_df.to_csv(
    'processed_reviews_comparison.csv', index=False, encoding='utf-8'
)

bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    columns=vectorizer.get_feature_names_out()
)
bow_df.to_csv('bow_representation.csv', index=False, encoding='utf-8')

print("Saved processed_reviews_comparison.csv")
print("Saved bow_representation.csv")

Average number of sentences per review: 10.84232
Average word count before preprocessing: 282.90412
Average word count after preprocessing: 124.25496
Saved processed_reviews_comparison.csv
Saved bow_representation.csv
